In [78]:
import torch
import numpy as np

x = torch.FloatTensor(np.array([[1 , 2 , 4],[3 , 4 , 5]])) #It is always better to use 32 Bits then the 64 as It takes half the computation and Speed
x

tensor([[1., 2., 4.],
        [3., 4., 5.]])

In [79]:
x[: , 1] = -1

In [80]:
x.relu_() #In place Operation

tensor([[1., 0., 4.],
        [3., 0., 5.]])

In [81]:
device = "cuda"
x = x.to(device=device)
x.device

device(type='cuda', index=0)

In [82]:
#Putting
learning_rate = 0.1
x = torch.tensor(5.0 , requires_grad=True)
for iteration in range(100):
    f = x**2
    f.backward()
    with torch.no_grad():
        x -=  learning_rate * x.grad
    x.grad.zero_()

In [83]:
#Implementing Linear Regression using Pytorch
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
import torch

housing = fetch_california_housing()
x = housing.data
y = housing.target

In [84]:
x_train , x_test , y_train , y_test = train_test_split(x , y , random_state=42)
x_train , x_valid , y_train , y_valid = train_test_split(x_train , y_train , random_state=42)

In [85]:
x_train = torch.FloatTensor(x_train)
x_valid = torch.FloatTensor(x_valid)
x_test = torch.FloatTensor(x_test)

In [86]:
means = x_train.mean(dim= 0 , keepdim=True)
std = x_train.std(dim=0 , keepdim=True)
x_train = (x_train - means) / std
x_test = (x_test - means ) / std
x_valid = (x_valid - means) / std

In [87]:
y_train = torch.FloatTensor(y_train.reshape(-1 , 1))
y_test = torch.FloatTensor(y_test.reshape(-1 , 1))
y_valid = torch.FloatTensor(y_valid.reshape(-1 , 1))

In [88]:
torch.manual_seed(42)
n_featurs = x_train.shape[1]
w = torch.randn((n_featurs , 1) , requires_grad=True) #Random weigth initiliation
b = torch.tensor(0.0 , requires_grad=True) #Bias Term

In [89]:
learning_rate = 0.4
epochs = 20
for iterations in range(epochs):
    y_pred = x_train @ w + b #Forward Pass 
    loss = ((y_pred - y_train)** 2).mean()
    loss.backward() #Backward Pass
    with torch.no_grad():
        b -= learning_rate * b.grad
        w -= learning_rate * w.grad
        b.grad.zero_()
        w.grad.zero_()
    print(f"Epochs {iterations + 1} / {epochs} , Loss {loss.item()}")

Epochs 1 / 20 , Loss 16.158456802368164
Epochs 2 / 20 , Loss 4.879360675811768
Epochs 3 / 20 , Loss 2.255225896835327
Epochs 4 / 20 , Loss 1.3307620286941528
Epochs 5 / 20 , Loss 0.9680696129798889
Epochs 6 / 20 , Loss 0.814268171787262
Epochs 7 / 20 , Loss 0.7417048811912537
Epochs 8 / 20 , Loss 0.7020705342292786
Epochs 9 / 20 , Loss 0.676592230796814
Epochs 10 / 20 , Loss 0.6577968001365662
Epochs 11 / 20 , Loss 0.6426153779029846
Epochs 12 / 20 , Loss 0.6297225952148438
Epochs 13 / 20 , Loss 0.6184943914413452
Epochs 14 / 20 , Loss 0.6085970997810364
Epochs 15 / 20 , Loss 0.5998218655586243
Epochs 16 / 20 , Loss 0.5920187830924988
Epochs 17 / 20 , Loss 0.5850692391395569
Epochs 18 / 20 , Loss 0.5788735151290894
Epochs 19 / 20 , Loss 0.5733454823493958
Epochs 20 / 20 , Loss 0.5684101581573486


In [90]:
#Prediction
x_new = x_test[:3]
with torch.no_grad():
    y_pred = x_new @ w + b

print(y_pred)

tensor([[0.8916],
        [1.6480],
        [2.6577]])


In [91]:
#Using Higher Level API
import torch.nn as nn

torch.manual_seed(42)
model = nn.Linear(in_features= n_featurs , out_features= 1)
model.weight

Parameter containing:
tensor([[ 0.2703,  0.2935, -0.0828,  0.3248, -0.0775,  0.0713, -0.1721,  0.2076]],
       requires_grad=True)

In [92]:
learning_rate = 0.4
optimizer = torch.optim.SGD(model.parameters() , lr=learning_rate)

loss = nn.MSELoss()

In [93]:
#Train_Our model 
def train_bgd(model , optimizer , x_train , y_train , criterion , n_epochs): #Batch GD
    for epochs in range(n_epochs):
        y_pred = model(x_train)
        loss = criterion(y_pred , y_train) #Criterion is often Refered as the Loss function
        loss.backward()
        optimizer.step() #This is to Update the weights 
        optimizer.zero_grad() #This is to make the Gradient zero
        print(f"Epoch {epochs + 1}/{n_epochs}, Loss: {loss.item()}")

In [ ]:
# BATCH GRADIENT DESCENT
torch.manual_seed(42)
model_bgd = nn.Sequential(
    nn.Linear(n_featurs, 50),
    nn.ReLU(),
    nn.Linear(50, 40),
    nn.ReLU(),
    nn.Linear(40, 1)
)

learning_rate = 0.01
optimizer_bgd = torch.optim.SGD(model_bgd.parameters(), lr=learning_rate)
loss_fn = nn.MSELoss()
train_bgd(model_bgd, optimizer_bgd, x_train, y_train, loss_fn, epochs)

Epoch 1/20, Loss: 5.045480251312256
Epoch 2/20, Loss: 4.68717622756958
Epoch 3/20, Loss: 4.346471309661865
Epoch 4/20, Loss: 4.020055294036865
Epoch 5/20, Loss: 3.706490993499756
Epoch 6/20, Loss: 3.4053335189819336
Epoch 7/20, Loss: 3.117288827896118
Epoch 8/20, Loss: 2.8437371253967285
Epoch 9/20, Loss: 2.5865440368652344
Epoch 10/20, Loss: 2.3477838039398193
Epoch 11/20, Loss: 2.12923002243042
Epoch 12/20, Loss: 1.9320870637893677
Epoch 13/20, Loss: 1.757125735282898
Epoch 14/20, Loss: 1.6041170358657837
Epoch 15/20, Loss: 1.4722553491592407
Epoch 16/20, Loss: 1.3601949214935303
Epoch 17/20, Loss: 1.2660672664642334
Epoch 18/20, Loss: 1.1877902746200562
Epoch 19/20, Loss: 1.1231715679168701
Epoch 20/20, Loss: 1.0700280666351318


In [96]:
from torch.utils.data import TensorDataset , DataLoader
import torch
import torch.nn as nn 


tensor_dataset = TensorDataset(x_train , y_train)
train_loader = DataLoader(tensor_dataset , batch_size=32 , shuffle=True , pin_memory=True)
torch.manual_seed(42)
model = nn.Sequential(
    nn.Linear(n_featurs , 50),
    nn.ReLU(),
    nn.Linear(50 , 40),
    nn.ReLU(),
    nn.Linear(40 , 1)
)
model = model.to(device="cuda")
optimizer = torch.optim.SGD(model.parameters() , lr = learning_rate)
loss = nn.MSELoss()

#Mini Batch Gradients
def train(model , optimizer , criterion , train_loader , n_epochs):
    model.train()
    for epochs in range(n_epochs):
        total_loss = 0.0
        for x_batch, y_batch in train_loader:
            x_batch , y_batch = x_batch.to(device = "cuda") , y_batch.to(device = "cuda")
            y_pred = model(x_batch)
            loss = criterion(y_pred , y_batch)
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
        mean_loss = total_loss / len(train_loader)
        print(f"Epoch {epochs +1} / {n_epochs} , loss:{mean_loss:.4f}")

train(model , optimizer , loss , train_loader , epochs)

Epoch 1 / 20 , loss:0.6958
Epoch 2 / 20 , loss:0.4480
Epoch 3 / 20 , loss:0.4039
Epoch 4 / 20 , loss:0.3843
Epoch 5 / 20 , loss:0.3708
Epoch 6 / 20 , loss:0.3648
Epoch 7 / 20 , loss:0.3556
Epoch 8 / 20 , loss:0.3520
Epoch 9 / 20 , loss:0.3488
Epoch 10 / 20 , loss:0.3461
Epoch 11 / 20 , loss:0.3412
Epoch 12 / 20 , loss:0.3381
Epoch 13 / 20 , loss:0.3358
Epoch 14 / 20 , loss:0.3315
Epoch 15 / 20 , loss:0.3319
Epoch 16 / 20 , loss:0.3287
Epoch 17 / 20 , loss:0.3271
Epoch 18 / 20 , loss:0.3248
Epoch 19 / 20 , loss:0.3293
Epoch 20 / 20 , loss:0.3229


In [ ]:
#Model Evaluation (Validation set)
def evaluate(model , data_loader , metrics_fn , aggregate_fn = lambda metrics : torch.sqrt(torch.mean(metrics))):
    model.eval()
    metrics = []
    with torch.no_grad():
        for x_batch , y_batch in data_loader:
            x_batch = x_batch.to(device = "cuda")
            y_batch = y_batch.to(device = "cuda")
            y_pred = model(x_batch)
            metric = metrics_fn(y_pred , y_batch)
            metrics.append(metric)
        return aggregate_fn(torch.stack(metrics)) #Metrics mean List of tensors here and the aggregate_fn mean to calculate the mean

valid_dataste = TensorDataset(x_valid ,y_valid)
valid_data_loader = DataLoader(valid_dataste , batch_size=32 , pin_memory= True)
valid_mse = evaluate(model , valid_data_loader , loss)
valid_mse

tensor(0.6554, device='cuda:0')

In [ ]:
def evaluate_train(model , data_loaders , metrics_fn , aggregate_fn = lambda metrics: torch.sqrt(torch.mean(metrics))):
    model.eval()
    metrics = []
    with torch.no_grad():
        for x_batch , y_batch in data_loaders:
            x_batch = x_batch.to(device = "cuda")
            y_batch = y_batch.to(device = "cuda")
            y_pred = model(x_batch)
            metric = metrics_fn(y_pred , y_batch)
            metrics.append(metric)
        return aggregate_fn(torch.stack(metrics))

train_dataset = TensorDataset(x_train , y_train)
train_data_loader = DataLoader(train_dataset , batch_size=32 , pin_memory= True)
train_rmse = evaluate_train(model , train_data_loader , loss)
train_rmse

tensor(0.5629, device='cuda:0')

In [2]:
#Non_sequential model
import torch
import torch.nn as nn
class WideandDeep(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.deep_stack = nn.Sequential(
            nn.Linear(n_features , 50) , nn.ReLU(),
            nn.Linear(50 , 40) , nn.ReLU()
        )
        self.output_layer = nn.Linear(n_features + 40 , 1) #N_features bhaneko chahi Input Layer zodney bhanera
    
    def forward(self , x):
        deep_output = self.deep_stack(x)
        wide_and_deep = torch.concat([x , deep_output] , dim=1)
        return self.output_layer(wide_and_deep)


In [3]:
#Image Classifier using PYtorch
import torchvision
import torchvision.transforms.v2 as T

toTensor = T.Compose([T.ToImage() , T.ToDtype(torch.float32 , scale=True)])

train_valid_datset = torchvision.datasets.FashionMNIST(root="datasets" , train=True , download=True , transform=toTensor)
test_dataset = torchvision.datasets.FashionMNIST(root="datasets" , train=False , download=True , transform=toTensor)

torch.manual_seed(42)
train_data , valid_data = torch.utils.data.random_split(train_valid_datset , [55000 , 5000])

100%|██████████| 26.4M/26.4M [00:04<00:00, 5.86MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 134kB/s]
100%|██████████| 4.42M/4.42M [00:02<00:00, 2.19MB/s]
100%|██████████| 5.15k/5.15k [00:00<?, ?B/s]


In [4]:
#Data_Loaders
from torch.utils.data import DataLoader

train_loader = DataLoader(train_data , batch_size=32 , shuffle=True)
valid_loader = DataLoader(valid_data , batch_size=32)
test_loader = DataLoader(test_dataset , batch_size= 32)


In [ ]:
#Classifier
class ImageClassifier(nn.Module):
    def __init__(self, n_inputs , n_classes):
        super().__init__()
        self.model = nn.Sequential(
            nn.Flatten(),
            nn.Linear(n_inputs , 300),
            nn.ReLU(),
            nn.Linear(300 , 200),
            nn.ReLU(),
            nn.Linear(200 , n_classes)
        )
    def forward(self , x):
        return self.model(x)

'Ankle boot'